In [ ]:
import random
import time
from datasets import load_dataset, Dataset, IterableDataset
from tqdm import tqdm # 사용자에게 진행 상황을 알려주는 장난스러운 도구!

# =============================================================================
# 💖 초보자를 위한 AI 코딩 실습: 금융 지식 쿼리 파악하기 💖
# =============================================================================
# 데이터셋 이름: nayohan/Sujet-Finance-Instruct-177k-ko
# 데이터셋 의미: 이 데이터셋은 17만 7천 건이 넘는 금융(Finance) 관련 질의응답(Question-Answering) 및
#               명령어 수행(Instruction-Tuning) 데이터입니다. 마치 AI가 '금융 전문가 인터뷰'를 받은 것처럼 훈련된 자료예요.
# 데이터 구조:
#   - system_prompt: AI가 어떤 역할(Role)을 맡을지 지시하는 큰 그림 (예: "당신은 금융 분석가입니다.")
#   - user_prompt: 사용자(학생님!)가 던지는 구체적인 질문 (예: "금리 인하의 영향은 무엇인가요?")
#   - answer: AI가 제공하는 정답/답변.
# 실습 목표: 이 데이터셋의 '숨겨진 보석'인 `task_type`을 분석하여, 사용자가 어떤 종류의 질문을 던졌는지 패턴을 파악하고, 나만의 프롬프트를 설계하는 능력을 키워봅시다!
# =============================================================================

# 🏷️ 상수 설정
DATASET_NAME = "nayohan/Sujet-Finance-Instruct-177k-ko"
SPLIT_NAME = "train"
SAMPLE_COUNT = 5 # 실습을 위해 상위 5개 샘플만 사용합니다! 너무 많은 데이터는 로딩 시간을 잡아먹어요.

# -----------------------------------------------------------------------------
# 🚀 1단계: 데이터 로드 (가장 빠르고 우아한 방법 찾기)
# -----------------------------------------------------------------------------

print("✨ [튜터 코멘트] 안녕하세요! 오늘은 금융 AI의 심장부를 탐험하는 날이에요. 데이터 로드는 가장 빠르고 안전한 방법을 찾아야 합니다!")
dataset = None

# 💡 핵심 목표: 스트리밍 로드(streaming=True)가 가장 빠르지만, 가끔 오류가 날 수 있어요.
# 그래서 try-except 구조를 사용해서 '속도'와 '안정성'을 모두 잡을 거예요!
try:
    print(f"\n🔍 시도 1/2: 스트리밍 모드로 데이터 로드 중... (최대 속도 모드)")
    start_time = time.time()
    # streaming=True는 메모리를 적게 쓰고, 데이터가 들어올 때마다 처리할 수 있게 해줘요.
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print(f"✅ 스트리밍 로드 성공! (소요 시간: {time.time() - start_time:.2f}초)")

except Exception as e:
    # 만약 스트리밍이 너무 복잡하거나 실패하면, 그냥 몇 개만 받아와서 천천히 진행합니다.
    print(f"⚠️ 스트리밍 로드 실패 또는 환경 문제 감지 ({e}). 대체 계획 실행!")
    print("✨ [튜터 코멘트] 괜찮아요! 환경에 따라 데이터 로드 방식이 달라지거든요. 안정적인 '일반 다운로드' 모드로 전환할게요.")
    try:
        # streaming=False로 설정하고, 로드 가능한 최소한의 데이터만 가져와요.
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME)
        print(f"✅ 일반 모드 로드 성공! (총 {dataset.num_rows}개 데이터셋 로드)")
    except Exception as e_fallback:
        print(f"❌ 치명적 오류 발생: 데이터 로드에 실패했습니다. 오류: {e_fallback}")
        exit()


# -----------------------------------------------------------------------------
# 📊 2단계: 데이터셋 구조 분석 및 샘플 추출
# -----------------------------------------------------------------------------

print("\n============================================================")
print("📚 2단계: 데이터 구조와 패턴 분석하기 (메타데이터 탐색)")
print("============================================================")

# 🧩 데이터의 기본 구조를 파악해 봅시다.
print("🔑 데이터셋의 주요 필드(Column) 목록:")
print(f"   - inputs (입력값): 문제나 배경 정보가 담겨요.")
print(f"   - user_prompt (사용자 질문): 우리가 실제로 질문할 내용.")
print(f"   - system_prompt (시스템 역할): AI의 페르소나를 설정해요. (예: '당신은 최고의 전문가')")
print(f"   - task_type (작업 유형): 이 데이터가 어떤 종류의 훈련을 받았는지 보여주는 분류 태그예요! (★가장 중요!)")
print(f"   - answer (답변): 최종적으로 나와야 하는 정답.")


# 🔢 데이터셋이 iterable한지 확인하고 샘플을 추출합니다. (필수 패턴 지키기!)
print(f"\n🔬 {DATASET_NAME}에서 가장 흥미로운 {SAMPLE_COUNT}개의 샘플을 골라볼게요.")

# 🌟 Streaming 여부에 따라 다른 코드를 실행합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 가능성이 높습니다.
    print("✅ 현재 데이터셋은 스트리밍(Streaming) 모드입니다. .take()를 사용해 샘플을 가져옵니다.")
    # 메모리 효율을 위해 리스트로 변환하는 대신, 반복자(iterator)로 사용합니다.
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)일 경우
    print("✅ 현재 데이터셋은 일반(Standard) 모드입니다. .take()를 사용해 샘플을 가져옵니다.")
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)

# ➡️ 리스트로 미리 변환하여 일관성 있게 순회합니다.
sample_data_list = list(sample_dataset_iterator)

# -----------------------------------------------------------------------------
# 💡 3단계: 초보자 맞춤 실습 - 금융 프롬프트 패턴 분류기 만들기
# -----------------------------------------------------------------------------

print("\n\n============================================================")
print("🌟 3단계: 실습 시작! 금융 질의응답 패턴 분석가 되기!")
print("============================================================")

print(f"🎯 목표: {SAMPLE_COUNT}개의 샘플을 분석하여, AI에게 가장 효과적인 프롬프트 구조를 찾아보세요.")
print("✨ [튜터 코멘트] 이 코드는 마치 데이터를 스캐닝해서 '질문의 의도'를 알아내는 패턴 인식기 같아요!")

# 💖 샘플 데이터 순회 및 분석
for i, sample in enumerate(sample_data_list):
    print("\n" + "=" * 50)
    print(f"   🤖 [분석 샘플 {i + 1}/{SAMPLE_COUNT}] - 데이터셋 분석 시간: {time.strftime('%H:%M:%S')}")

    # 📥 필요한 핵심 정보만 깔끔하게 추출합니다.
    task = sample.get('task_type', '미분류')
    user_q = sample.get('user_prompt', '정보 없음')
    system_r = sample.get('system_prompt', '역할 미지정')
    answer_text = sample.get('answer', '답변 없음')

    # 📈 분석 결과를 구조화하여 출력합니다.
    print(f"💡 분석 결과 [Task Type]: '{task}'")
    print(f"   👤 User Query (사용자 질문): {user_q[:50]}...")
    print(f"   🎭 System Role (AI 역할): {system_r[:50]}...")
    print("-------------------------------------------------")
    
    # 🛠️ 창의적 분석: task_type에 따라 요구되는 행동 패턴을 유추합니다.
    print("🧠 [튜터 인사이트] 이 샘플의 패턴은 다음과 같아요:")
    if 'qa' in task.lower():
        print(f"   👉 📝 **[QA 패턴]:** 사용자는 질문을 던지고, AI는 정확하고 간결한 정보를 필요로 합니다. (정답 근거 제시 필수!)")
    elif 'generation' in task.lower():
        print(f"   👉 🌐 **[생성 패턴]:** AI는 단순히 답변만 할 것이 아니라, 새로운 보고서나 설명을 '만들어내야' 합니다. (창의적 설명력 필요!)")
    elif 'classification' in task.lower():
        print(f"   👉 🏷️ **[분류 패턴]:** 이 데이터는 분류 문제일 가능성이 높습니다. (특정 키워드를 잡아 특정 카테고리로 분류하는 연습이 중요합니다!)")
    else:
        print("   👉 ✨ **[기타 패턴]:** 일반적인 지식 전달 또는 요약 요청 패턴입니다.")

    # ✅ 최종적인 AI에게 전달할 '완성된 프롬프트' 형태로 조합하는 시뮬레이션
    print("\n✅ [최종 프롬프트 설계 시뮬레이션]:")
    print(f"   << 시스템 지침>>: {system_r}")
    print(f"   << 사용자 요청>>: {user_q}")
    print(f"   << 기대되는 답변>>: {answer_text[:50]}...")
    print("-------------------------------------------------")


print("\n\n🎉 🎊 축하합니다! 데이터를 성공적으로 탐색하셨습니다! 🎊 🎉")
print("✨ [튜터 코멘트] 단순히 코드를 실행하는 것을 넘어, 데이터가 가진 '의미'와 '구조'를 파악하는 것이 바로 AI 개발자의 핵심 역량입니다! 👍")